# AEP tutorial: a swiss roll in four dimensions

This notebook does four things:

1. It generates a noisy **swiss roll in R⁴**. The 2D sheet is rolled up in coordinates 1–2 and
   curled again in coordinates 3–4, then randomly rotated.
2. It fits **AEP** with default parameters.
3. It shows the data and the fit as interactive **plotly** 3D scatter plots.
4. It embeds new points with the fitted model and saves and reloads the model.

Install the tutorial extras first: `uv sync --extra tutorial` or `pip install -e ".[tutorial]"`.

In [1]:
import time

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from aep import AEPModel, fit_aep

pio.renderers.default = "notebook_connected"   # plotly.js loaded from a CDN keeps the notebook small

## 1. Generate the data

The swiss roll is a flat 2D sheet. One direction runs along the spiral `(t cos t, t sin t)`; the
other is a straight height `h`. Here the height is also curled, through coordinates 3–4, as a
270° arc of radius 12, so the sheet is rolled up in two directions at once. Lengths along both
directions are kept, so the sheet is still intrinsically flat.

Isotropic noise (σ = 0.2) is added and the result is randomly rotated in R⁴. No single coordinate
pair then shows the roll. We color points by **arc length along the spiral**, the sheet's first
intrinsic coordinate.

In [2]:
def swiss_roll_4d(n=10_000, sigma=0.2, seed=0, radius=12.0, degrees=270.0):
    """Swiss roll in R^4: spiral (t cos t, t sin t) in coordinates 1-2, height curled through
    coordinates 3-4 as an arc of the given radius; noise, then a random rotation of R^4.
    Returns (X, arc length along the spiral, height, unrotated noisy coordinates)."""
    rng = np.random.default_rng(seed)
    t = rng.uniform(1.5 * np.pi, 4.5 * np.pi, n)                 # spiral parameter
    rng.uniform(0, 21, n)                                        # (unused draw, keeps the stream fixed)
    half = np.radians(degrees) / 2
    theta = rng.uniform(-half, half, n)                          # angle along the curled height
    X = np.column_stack([t * np.cos(t), t * np.sin(t),
                         radius * np.cos(theta), radius * np.sin(theta)])
    Z = X + sigma * rng.standard_normal(X.shape)                 # noisy, unrotated
    R, _ = np.linalg.qr(rng.standard_normal((4, 4)))             # random rotation
    arc = 0.5 * (t * np.sqrt(1 + t**2) + np.arcsinh(t))         # arc length along the spiral
    return Z @ R.T, arc, radius * theta, Z


X, arc, height, Z = swiss_roll_4d()
print(X.shape)

(10000, 4)


A helper for the plotly read-outs: a 3D scatter colored by arc length, with equal scale on
all axes.

In [3]:
def scatter3d(Y, color, title, prefix, colorbar="arc length"):
    fig = go.Figure(go.Scatter3d(
        x=Y[:, 0], y=Y[:, 1], z=Y[:, 2], mode="markers",
        marker=dict(size=1.5, color=color, colorscale="Viridis",
                    colorbar=dict(title=colorbar, thickness=12))))
    fig.update_layout(title=title, width=650, height=550, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_title=f"{prefix} 1",
                                 yaxis_title=f"{prefix} 2", zaxis_title=f"{prefix} 3"))
    return fig

### The raw data

The unrotated coordinates 1–2 show the spiral. The rotated data `X` that AEP sees mix all four
coordinates, so any three of them give a tangle.

In [4]:
go.Figure(go.Scatter(x=Z[:, 0], y=Z[:, 1], mode="markers",
                     marker=dict(size=3, color=arc, colorscale="Viridis"))).update_layout(
    title="Unrotated coordinates 1-2: the spiral", width=500, height=480,
    xaxis_title="dimension 1", yaxis_title="dimension 2", yaxis_scaleanchor="x")

In [5]:
scatter3d(X[:, :3], arc, "Rotated input: coordinates 1-3 of X", "X")

## 2. Fit AEP

`fit_aep` uses the defaults here. `X` has D = 4 features, so the working space is all four PCs.
Neighborhoods have `k = 20` points and training runs for 5,000 steps. It returns the 3D
coordinates `Y`, the fitted `model` and an `info` dictionary.

In [6]:
t0 = time.perf_counter()
Y, model, info = fit_aep(X, seed=0)
print(f"fit in {time.perf_counter() - t0:.0f} s")
print("settings:", {k: info["settings"][k] for k in ("n_pca", "k", "steps", "batch_size", "lambda_s")})
print("PCA variance retained:", round(info["variance_retained"], 4))

fit in 17 s
settings: {'n_pca': 4, 'k': 20, 'steps': 5000, 'batch_size': 256, 'lambda_s': 5.0}
PCA variance retained: 1.0


## 3. Plotly read-outs of the embedding

The AEP coordinates put the noisy 4D roll on a smooth surface in 3D. Color is arc length along
the spiral; rotate the plot to see the surface.

In [7]:
scatter3d(Y, arc, "AEP embedding, colored by arc length along the spiral", "AEP")

Coloring by the second intrinsic coordinate, the curled height, shows that both directions of
the sheet are kept.

In [8]:
scatter3d(Y, height, "AEP embedding, colored by height", "AEP", colorbar="height")

## 4. New points, saving and loading

The model is a function, so it can embed points it was not trained on. Here we draw a fresh swiss
roll with a different seed. The data are rotated with a different random rotation, so we put the
new points into the original frame with the first roll's rotation before embedding them. Then we
save and reload the model.

In [9]:
_, arc_new, _, Z_new = swiss_roll_4d(n=2_000, seed=1)
R = np.linalg.lstsq(Z, X, rcond=None)[0]          # recover the first roll's rotation (X = Z R)
X_new = Z_new @ R
Y_new = model(X_new)

fig = scatter3d(Y, arc, "Training points (small) and new points (large)", "AEP")
fig.data[0].marker.opacity = 0.15
fig.add_trace(go.Scatter3d(x=Y_new[:, 0], y=Y_new[:, 1], z=Y_new[:, 2], mode="markers",
                           marker=dict(size=3, color=arc_new, colorscale="Viridis"), showlegend=False))
fig

In [10]:
model.save("aep_swiss_roll.pt")
reloaded = AEPModel.load("aep_swiss_roll.pt")
print("max difference after reload:", np.abs(reloaded(X) - Y).max())

max difference after reload: 0.0
